In [107]:
from pathlib import Path
from enum import Enum
import hashlib
import csv
import json
import docx
import pypdf

class FileType(Enum):
    DOCX = ".docx"
    PDF = ".pdf"
    CSV = ".csv"
    JSON = ".json"
    TXT = ".txt"
    MD = ".md"

def string_to_hash_id(source_file: str, source_type: str, block_id: int, text: str) -> str:
    key = f"{source_file}::{source_type}::{block_id}::{text}"
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

def parse_document(file_path: str | Path) -> str:
    path = Path(file_path)
    ext = path.suffix.lower()
    file_str = str(path)
    blocks: list[str] = []

    try:
        file_type = FileType(ext)
    except ValueError:
        raise ValueError(f"Unsupported format: {ext}")

    match file_type:
        # 1. DOCX
        case FileType.DOCX:
            doc = docx.Document(path)
            text = "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
            if text:
                blocks.append(text)

        # 2. PDF
        case FileType.PDF:
            reader = pypdf.PdfReader(path)
            page_texts = [
                page.extract_text().strip()
                for page in reader.pages
                if page.extract_text() and page.extract_text().strip()
            ]
            full_text = "\n\n".join(page_texts)
            if full_text:
                blocks.append(full_text)

        # 3. CSV
        case FileType.CSV:
            with open(path, mode="r", encoding="utf-8", errors="ignore") as f:
                reader = csv.reader(f)
                header = next(reader, None)

                if header:
                    for row in reader:
                        if row:
                            paired_row = [
                                f"{h.strip()}: {v.strip()}"
                                for h, v in zip(header, row)
                            ]
                            blocks.append(" | ".join(paired_row))

        # 4. JSON
        case FileType.JSON:
            with open(path, mode="r", encoding="utf-8", errors="ignore") as f:
                data = json.load(f)
                if isinstance(data, list):
                    blocks = [json.dumps(item, ensure_ascii=False) for item in data]
                else:
                    blocks = [json.dumps(data, indent=2, ensure_ascii=False)]

        # 5. TXT / MD
        case FileType.TXT | FileType.MD:
            with open(path, mode="r", encoding="utf-8", errors="ignore") as f:
                content = f.read().strip()
                if content:
                    blocks.append(content)

    output_blocks = []
    for i, block in enumerate(blocks):
        output_blocks.append(
            {
                "id": string_to_hash_id(file_str, ext, i, block),
                "source_file": file_str,
                "source_type": ext,
                "text": block,
            }
        )

    return output_blocks

In [114]:
parsed_blocks = parse_document('../Sample Files/sample.txt')
for block in parsed_blocks:
    print(f"{block["text"]}\n")

Server maintenance log for the Aster Cloud cluster, region eu-west-3, covering the week of March 3rd.

On Monday morning, engineer Priya Nandakumar rotated the TLS certificates on the ingress gateway after a routine expiry check flagged three services running under 200 hours of validity remaining.

By Tuesday, the billing-worker pod began throwing intermittent OOMKilled events, traced back to a memory leak in the retry queue introduced in build 4.12.0; a hotfix was rolled out that evening.

Wednesday saw a scheduled failover test of the primary Postgres instance to its eu-west-3b replica, which completed in 43 seconds with zero dropped writes.

Thursday's standup covered a customer-reported latency spike on the search endpoint, later identified as a missing index on the `orders.created_at` column after a schema migration two weeks prior.

On Friday, the security team ran an automated dependency scan and flagged CVE-2024-31331 in an outdated image-processing library, which was patched b

In [ ]:
from RAG.Pipeline.Retry_Handler import call_safe_function
from google import genai
from dotenv import load_dotenv
import numpy as np

load_dotenv()
client = genai.Client()

# Normalized vector helps with cosine similarity.
def normalize_embedding(embedding):
    print("Normalizing Semantic Embeddings...")
    vectors = np.array([e.values for e in embedding.embeddings])
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

def embed(texts: list[str], model = "gemini-embedding-001", config=None):
    print(f"[embed] {len(texts)} texts, 1 request")
    return call_safe_function(
        client.models.embed_content,
        model = model,
        contents = texts,
        config = config or {"output_dimensionality": 768},
    )

def generate_response(prompt: str, model="gemini-3-flash", config=None):
    """
    config example: {
        "system_instruction": "You are a helpful assistant.",
        "temperature": 0.7,
        "max_output_tokens": 1024,
    }
    """
    return call_safe_function(
        client.models.generate_content,
        model = model,
        contents = prompt,
        config = config,
    )

